# Smart SOC Wazuh - Baseline (All Sources)

Notebook ini menjalankan pipeline full source:
1. Build dataset gabungan semua sumber
2. Train baseline RandomForest
3. Baca metrics dan visualisasi confusion matrix

In [ ]:
import json
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
ROOT = Path('.')
print('Working dir:', ROOT.resolve())
print('Build script exists:', (ROOT / 'build_train_ready_all_sources.py').exists())
print('Train script exists:', (ROOT / 'train_baseline_all_sources.py').exists())

## 1) Build Train Ready (All Sources)

In [ ]:
subprocess.run(['python3', 'build_train_ready_all_sources.py'], check=True)

In [ ]:
train_ready_path = Path('parsed/train_ready_all_sources.csv')
df = pd.read_csv(train_ready_path)
print('Shape:', df.shape)
display(df.head(3))
print('\nLabel distribution:')
print(df['label'].value_counts())

## 2) Train Baseline Model (All Sources)

In [ ]:
subprocess.run(['python3', 'train_baseline_all_sources.py'], check=True)

## 3) Read Metrics

In [ ]:
metrics_path = Path('parsed/model_all_sources/metrics_all_sources.json')
with metrics_path.open('r', encoding='utf-8') as f:
    metrics = json.load(f)

print('Input      :', metrics['input'])
print('Train/Test :', metrics['train_size'], '/', metrics['test_size'])
print('Accuracy   :', round(metrics['accuracy'], 4))
print('Labels     :', metrics['labels'])

In [ ]:
report = pd.DataFrame(metrics['classification_report']).T
display(report)

## 4) Confusion Matrix Plot

In [ ]:
labels = metrics['labels']
cm = np.array(metrics['confusion_matrix'])

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')

ax.set_xticks(np.arange(len(labels)))
ax.set_yticks(np.arange(len(labels)))
ax.set_xticklabels(labels)
ax.set_yticklabels(labels)
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_title('Confusion Matrix (All Sources)')

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', color='black')

fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 5) Saved Artifacts

- Model: `parsed/model_all_sources/baseline_rf_all_sources.joblib`
- Metrics: `parsed/model_all_sources/metrics_all_sources.json`
- Data gabungan: `parsed/train_ready_all_sources.csv`